# Fine-tune a small LLM for agent tool calling

This trains a LoRA adapter on Qwen2.5-1.5B, serves it with vLLM, and runs the agent's 38-question benchmark on the base and fine-tuned models.

**Before you start:** choose **Runtime > Change runtime type > T4 GPU**, then run the cells in order, one at a time. Expect about 1.5 to 3 hours in total.

Progress is saved to your Google Drive. If Colab disconnects, reconnect and run all the cells again from the top. Training and the benchmark both pick up where they left off.

## 1. Check for a GPU

In [1]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, then run this again."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

Tesla T4 15.6 GB


## 2. Save progress to Google Drive

Allow access when asked. Everything goes in one folder named `agent-lora-finetune`.

In [2]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/agent-lora-finetune"
os.makedirs(DRIVE, exist_ok=True)
os.environ["DRIVE"] = DRIVE
print("Saving to", DRIVE)

Mounted at /content/drive
Saving to /content/drive/MyDrive/agent-lora-finetune


## 3. Install the project and the agent

The agent is pinned to the commit that produced the Gemini score, so all the models get the same prompt, tools and benchmark.

In [3]:
%cd /content
!rm -rf agent-lora-finetune
!git clone -q https://github.com/ykrubha/agent-lora-finetune.git
%cd /content/agent-lora-finetune
!git clone -q https://github.com/ykrubha/agentic-data-analyst.git agent
!git -C agent checkout -q e82e4f7a52f01fe96d8d9d37a49a8ae1161fc8fa
!pip install -q -e agent -e . pytest "transformers>=4.56" "peft>=0.12" "accelerate>=0.33"
!git -C agent log -1 --format='Agent commit: %h %s'
import sys
# This kernel started before the install, so add the new packages to its path directly.
sys.path[:0] = ["/content/agent-lora-finetune/src", "/content/agent-lora-finetune/agent/src"]


/content
/content/agent-lora-finetune
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for analyst-agent (pyproject.toml) ... done
  Building editable for agent-lora-finetune (pyproject.toml) ... done
Agent commit: e82e4f7 Add benchmark results


## 4. Generate the training data

This runs the agent's real tools on 8 synthetic databases. None of them are the benchmark database, and two are held out for validation.

In [5]:
!pip install -q "sqlglot==30.29.0"
!python scripts/generate_data.py

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires sqlglot<25.21,>=23.4, but you have sqlglot 30.29.0 which is incompatible.
{
 "train": 505,
 "validation": 62,
 "skipped": 2,
 "train_domains": {
  "logistics": 89,
  "airline": 78,
  "retail": 111,
  "energy": 75,
  "university": 60,
  "hospital": 92
 },
 "train_kinds": {
  "multi_step": 84,
  "aggregation": 181,
  "join": 102,
  "python": 84,
  "refusal": 54
 },
 "validation_kinds": {
  "refusal": 6,
  "multi_step": 10,
  "join": 14,
  "aggregation": 22,
  "python": 10
 }
}


## 5. Check the real tokenizer before training

This stops here if Qwen's chat template breaks the training labels. The printed text should show only tool calls, with no system prompt or tool results.

In [6]:
import json, sys
import numpy as np
from transformers import AutoTokenizer
from finetune.train import IGNORE, encode_example, load_jsonl

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
rows = load_jsonl("data/train.jsonl")
encoded = [encode_example(row, tokenizer, 100000) for row in rows]
lengths = [len(item["input_ids"]) for item in encoded]
share = np.mean([np.mean(np.array(item["labels"]) != IGNORE) for item in encoded])
print(f"{len(rows)} examples. Tokens: median {int(np.median(lengths))}, 95th percentile {int(np.percentile(lengths, 95))}, longest {max(lengths)}.")
print(f"Over the 3072 token limit: {sum(n > 3072 for n in lengths)}. Share of tokens used as targets: {share:.1%}.")

sample = encoded[0]
targets = tokenizer.decode([i for i, label in zip(sample["input_ids"], sample["labels"]) if label != IGNORE])
print("\n--- Training targets for one example ---\n" + targets[:1200])
assert "<tool_call>" in targets and "<tool_response>" not in targets and "careful data analyst" not in targets
print("\nTokenizer check passed.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

505 examples. Tokens: median 1636, 95th percentile 2018, longest 2234.
Over the 3072 token limit: 0. Share of tokens used as targets: 9.6%.

--- Training targets for one example ---
<tool_call>
{"name": "list_tables", "arguments": {}}
</tool_call><|im_end|>
<tool_call>
{"name": "describe_table", "arguments": {"table": "Shipment"}}
</tool_call><|im_end|>
<tool_call>
{"name": "run_sql", "arguments": {"query": "SELECT strftime('%Y-%m', \"ShipDate\") AS label, SUM(\"Charge\") AS metric FROM \"Shipment\" GROUP BY label ORDER BY metric DESC LIMIT 2", "save_as": "df"}}
</tool_call><|im_end|>
<tool_call>
{"name": "final_answer", "arguments": {"answer": "Total charge was highest in 2024-04.", "value": "2024-04", "answerable": true, "assumptions": []}}
</tool_call><|im_end|>


Tokenizer check passed.


In [8]:
!pip uninstall -y -q torchao

## 6. Train

This should take about 20 to 40 minutes on a T4. Paste the printed `validation_loss` lines when you share results.

In [ ]:
!python scripts/train_lora.py --model Qwen/Qwen2.5-1.5B-Instruct --output "$DRIVE/adapter" --epochs 3

Loading weights: 100% 338/338 [00:06<00:00, 50.01it/s]
Training 4,358,144 of 1,548,072,448 parameters on cuda. 505 train and 62 validation examples (0 too long and dropped).
epoch 1 step 20/189 loss 0.2979 202s
epoch 1 step 40/189 loss 0.1782 422s
epoch 1 step 60/189 loss 0.1250 631s
{'epoch': 1, 'train_loss': 0.1199, 'validation_loss': 0.0143}


## 7. Start vLLM with both models

vLLM gets its own environment so it can't conflict with Colab's packages. This takes several minutes the first time, and the server keeps running in the background.

In [ ]:
!python -m venv /content/vllm-env
!/content/vllm-env/bin/pip install -q vllm

In [ ]:
import subprocess, time, urllib.request, json, os
log = open("/content/vllm.log", "w")
server = subprocess.Popen([
    "/content/vllm-env/bin/vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct",
    "--served-model-name", "base",
    "--enable-lora", "--max-lora-rank", "16",
    "--lora-modules", f"analyst={os.environ['DRIVE']}/adapter/best",
    "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
    "--max-model-len", "8192", "--gpu-memory-utilization", "0.85", "--dtype", "float16",
    "--override-generation-config", json.dumps({"temperature": 0.0}),
    "--port", "8000",
], stdout=log, stderr=subprocess.STDOUT)

for _ in range(120):
    if server.poll() is not None:
        break
    try:
        reply = json.load(urllib.request.urlopen("http://localhost:8000/v1/models", timeout=3))
        print("Ready:", [m["id"] for m in reply["data"]])
        break
    except Exception:
        time.sleep(5)
else:
    print("Timed out.")
if server.poll() is not None or not os.path.exists("/content/vllm.log"):
    print("The server stopped. Paste this log when asking for help:")
print(subprocess.run(["tail", "-n", "15", "/content/vllm.log"], capture_output=True, text=True).stdout)

## 8. Try one step on each model

A good result prints a tool call such as `list_tables`. The base model may print plain text instead, which is one of the failures we're measuring.

In [ ]:
import os, sys
os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
from analyst_agent.agent import SYSTEM_PROMPT
from analyst_agent.llm import make_provider
from analyst_agent.tools import TOOL_SCHEMAS

for name in ("base", "analyst"):
    reply = make_provider("openai", name).complete(
        SYSTEM_PROMPT, [{"role": "user", "content": "Which genre generated the most revenue?"}], TOOL_SCHEMAS)
    print(f"{name}: tool calls = {reply.tool_calls}\n  text = {reply.text[:200]!r}\n  output tokens = {reply.output_tokens}\n")

## 9. Run the benchmark on both models

This should take about 30 to 90 minutes. Each question prints `PASS` or `FAIL`. If Colab disconnects, reconnect and run everything again, and `--resume` will skip the finished questions.

Failures marked `ERROR` are server problems, not wrong answers, and no score gets written while any remain.

In [ ]:
%cd /content/agent-lora-finetune/agent
!python scripts/download_data.py
!OPENAI_BASE_URL=http://localhost:8000/v1 python -m eval.run_eval --provider openai --model base --max-seconds 600 --resume --out "$DRIVE/results/base"

In [ ]:
!OPENAI_BASE_URL=http://localhost:8000/v1 python -m eval.run_eval --provider openai --model analyst --max-seconds 600 --resume --out "$DRIVE/results/analyst"

## 10. Compare the results

This includes the earlier Gemini score from the agent's repo. Copy the printed table into the chat, and download the zip file for the repo.

In [ ]:
%cd /content/agent-lora-finetune
!python scripts/compare.py --run "Qwen 1.5B base=$DRIVE/results/base/benchmark.json" --run "Qwen 1.5B + LoRA=$DRIVE/results/analyst/benchmark.json" --run "Gemini 3.5 Flash Lite=agent/results/benchmark.json" --out "$DRIVE/results/comparison.md"
!cp "$DRIVE/adapter/training_report.json" "$DRIVE/results/" 2>/dev/null
!cp data/summary.json "$DRIVE/results/training_data_summary.json"
!cp /content/vllm.log "$DRIVE/results/vllm.log"
!cd "$DRIVE" && rm -f results.zip && zip -qr results.zip results adapter/best/adapter_config.json adapter/best/adapter_model.safetensors -x "results/vllm.log"
!ls -la "$DRIVE/results.zip"
from google.colab import files
files.download(f"{DRIVE}/results.zip")